# semialg demo

This notebook is a compact tour of the public API. It emphasizes exact mathematical returns, optional structured metadata, real-algebraic function semantics, exact algebraization, stable formula simplification, CAD variable ordering, and CAD-driven integration.

Every code cell is intended to run top-to-bottom without manual edits. The repository test suite executes the notebook cells as a contract so the demo stays synchronized with the package.


## 1. Setup

Use exact SymPy symbols. The notebook imports the installed package directly and then exercises the public API throughout.

In [1]:
import sympy as sp
import semialg

from semialg import (
    IntervalRegion,
    ParametricRegion,
    cad,
    equivalent,
    find_instance,
    function_domain,
    function_range,
    implies,
    integrate_over_region,
    is_satisfiable,
    reduce_region_integral,
    region_active_boundary_strata,
    region_boundary,
    semialgebraic_measure,
    semialgebraic_minimize,
    simplify_boole,
)
from semialg.solve.zero_dimensional import solve_zero_dimensional_system

x, y, z = sp.symbols("x y z", real=True)
t = sp.Symbol("t", real=True)


## 2. Exact decision problems

Semialgebraic formulas are Boolean combinations of polynomial equalities and inequalities. Decision functions return the mathematical Boolean answer by default.


In [2]:
disk_quadrant = (x**2 + y**2 <= 1) & (x > 0) & (y > 0)

assert is_satisfiable(disk_quadrant, [x, y]) is True
assert implies(x > 1, x**2 > 1, [x]) is True
assert equivalent(x**2 <= 1, (x >= -1) & (x <= 1), [x]) is True

print("disk quadrant satisfiable:", is_satisfiable(disk_quadrant, [x, y]))


disk quadrant satisfiable: True


When evidence or diagnostics matter, request the structured result explicitly with `return_result=True`.


In [3]:
implication = implies(x >= 0, x > 0, [x], return_result=True)
assert implication.valid is False
assert implication.counterexample is not None
print("counterexample:", implication.counterexample)


counterexample: {x: 0}


## 3. Direct mathematical returns

Primary operations return the mathematical object by default. Result wrappers are opt-in metadata containers.


In [4]:
region_formula = cad((x >= -1) & (x <= 2), [x])
minimum = semialgebraic_minimize((x - 2)**2, variables=[x])
instances = find_instance(sp.Eq(x**2, 2), [x], count=2)
points = solve_zero_dimensional_system([x**2 - 1], variables=[x])

assert region_formula == ((x >= -1) & (x <= 2))
assert minimum == [0, [{x: 2}]]
assert len(instances) == 2
assert set(points) == {(-1,), (1,)}

print("CAD formula:", region_formula)
print("minimum [value, optimizers]:", minimum)
print("instances:", instances)
print("zero-dimensional points:", points)


CAD formula: (x >= -1) & (x <= 2)
minimum [value, optimizers]: [0, [{x: 2}]]
instances: ({x: -sqrt(2)}, {x: sqrt(2)})
zero-dimensional points: ((-1,), (1,))


The same operations expose certification and computational details when requested.


In [5]:
cad_result = cad((x >= -1) & (x <= 2), [x], return_result=True)
opt_result = semialgebraic_minimize((x - 2)**2, variables=[x], return_result=True)

assert cad_result.formula == region_formula
assert opt_result.value == 0
assert opt_result.attained is True
assert opt_result.certified is True
print(opt_result.method)


exact_kkt_active_set+cad_decision_certificate


## 4. Real-root semantics and the shared function-graph layer

`semialg` respects the distinction between SymPy's ordinary principal-branch `Pow` and an explicit real root. For real-valued function analysis,

- `x**Rational(1, 3)` is the principal complex cube root restricted to where it is real;
- `real_root(x, 3)` is the real cube-root function on all real inputs.

`function_domain` and `function_range` share an internal semialgebraic graph representation, so nested radicals and common semialgebraic expression forms use one set of branch/domain semantics.


In [6]:
principal_domain = function_domain(x ** sp.Rational(1, 3), [x])
real_cube_domain = function_domain(sp.real_root(x, 3), [x])
nested_domain = function_domain(sp.sqrt(1 - sp.sqrt(x)), [x])

assert principal_domain == (x >= 0)
assert real_cube_domain is sp.true
assert equivalent(nested_domain, (x >= 0) & (x <= 1), [x])

print("principal cube-root domain:", principal_domain)
print("real cube-root domain:", real_cube_domain)
print("nested radical domain:", nested_domain)

heaviside_range = function_range(sp.Heaviside(x, sp.Rational(1, 3)), variables=[x], value_symbol=t)
assert equivalent(heaviside_range, sp.Or(sp.Eq(t, 0), sp.Eq(t, sp.Rational(1, 3)), sp.Eq(t, 1)), [t])


principal cube-root domain: x >= 0
real cube-root domain: True
nested radical domain: x**2 - x <= 0


## 5. Exact algebraization of selected transcendental ranges

The graph of `sin`, `exp`, or `cosh` is not semialgebraic. Nevertheless, some range problems admit an **exact finite reduction** to a semialgebraic problem.

For commensurate trigonometric polynomials, semialg introduces a common angle with `s = sin(u)`, `c = cos(u)`, and `s**2 + c**2 = 1`. For commensurate exponential/hyperbolic expressions it uses `q = exp(g*x) > 0`. Unsupported or noncommensurate cases are declined rather than relaxed.


In [7]:
trig_range = function_range(sp.sin(x) + sp.cos(2*x), variables=[x], value_symbol=t)
exp_range = function_range(sp.exp(x) + sp.exp(-x), variables=[x], value_symbol=t)
hyperbolic_range = function_range(sp.cosh(2*x), variables=[x], value_symbol=t)

assert equivalent(trig_range, (t >= -2) & (t <= sp.Rational(9, 8)), [t])
assert equivalent(exp_range, t >= 2, [t])
assert equivalent(hyperbolic_range, t >= 1, [t])

print("sin(x) + cos(2x):", trig_range)
print("exp(x) + exp(-x):", exp_range)
print("cosh(2x):", hyperbolic_range)


sin(x) + cos(2x): (t >= -2) & (t <= 9/8)
exp(x) + exp(-x): t >= 2
cosh(2x): t >= 1


Structured range results identify the exact transformation used.


In [8]:
trig_details = function_range(
    sp.sin(x) + sp.cos(2*x), variables=[x], value_symbol=t, return_result=True
)
assert trig_details.method == "algebraized_commensurate_trigonometric"
assert trig_details.infimum == -2
assert trig_details.supremum == sp.Rational(9, 8)
print(trig_details.method, trig_details.diagnostics["algebraization"])


algebraized_commensurate_trigonometric commensurate_trigonometric


## 6. Stable canonical formula simplification

The simplifiers aim for a deterministic, idempotent **canonical form** on the supported polynomial fragment: polynomial atoms are normalized for scalar/sign orientation, repeated zero-set multiplicities are removed, and semantic redundancy can be eliminated with guarded CAD reasoning.

This is intentionally a stable canonicalization target, not a claim to find a globally smallest Boolean formula.


In [9]:
from semialg.simplify import simplify_semialgebraic_formula

first = simplify_boole((2*x - 2*y > 0) & (x > -1), [x, y], semantic=False)
second = simplify_boole((y - x < 0) & (x > -1), [x, y], semantic=False)
zero_set = simplify_semialgebraic_formula(
    sp.Eq(6*(x - 1)**4*(x + 2)**2, 0), implication_minimize=False
)

assert first == second == ((x > -1) & (x - y > 0))
assert zero_set == sp.Eq(x**2 + x - 2, 0)
assert simplify_boole(first, [x, y], semantic=False) == first

print(first)
print(zero_set)


(x > -1) & (x - y > 0)
Eq(x**2 + x - 2, 0)


## 7. CAD variable ordering matters

CAD is highly sensitive to variable order. Equivalent orders can differ dramatically in projection degree, coefficient growth, root isolation, cell count, memory use, and runtime. Semialg has variable-order heuristics, but a mathematically informed order can still be valuable.

Reordering is only valid where the logical problem permits it: variables must not be moved across alternating quantifier blocks.


In [10]:
from semialg.heuristics import suggest_cad_variable_order

order_polynomials = [y, y - 1, x - y**2, x - y]
order_info = suggest_cad_variable_order(order_polynomials, [x, y], strategy="projection")
print("suggested CAD order:", order_info.order)
print("score:", order_info.score)

suggested CAD order: (x, y)
score: (7, 12)


## 8. CAD-driven general region integration

The integration reducer can search alternative cylindrical orders and convert certified full-dimensional CAD cells into nested exact integrals. This avoids privileging the caller's coordinate order.

For the region

\[
0\le y\le1,\qquad y^2\le x\le y,
\]

using `x` first introduces a square-root boundary. The integration planner instead selects `(y, x)`, yielding polynomial bounds.


In [11]:
condition = (y >= 0) & (y <= 1) & (x >= y**2) & (x <= y)
reduced = reduce_region_integral(1, condition, [x, y])
piece = reduced.pieces[0]
value = integrate_over_region(1, condition, [x, y])

assert reduced.method == "coordinate_permuted_cylindrical_integration"
assert piece.diagnostics["integration_variable_order"] == (y, x)
assert piece.limits == ((x, y**2, y), (y, 0, 1))
assert value == sp.Rational(1, 6)

print("selected integration order:", piece.diagnostics["integration_variable_order"])
print("nested limits:", piece.limits)
print("exact integral:", value)


selected integration order: (y, x)
nested limits: ((x, y**2, y), (y, 0, 1))
exact integral: 1/6


## 9. Measure, topology, and Boolean regions

Boolean combinations are interpreted geometrically as selected CAD cells, so internal seams are not mistaken for boundaries.


In [12]:
area = semialgebraic_measure(x**2 + y**2 <= 1, [x, y])
touching_union = sp.Or(
    sp.And(x >= 0, x <= 1),
    sp.And(x >= 1, x <= 2),
)
boundary = region_boundary(touching_union, [x])

assert area == sp.pi
assert equivalent(boundary, sp.Or(sp.Eq(x, 0), sp.Eq(x, 2)), [x])
print("unit-disk area:", area)
print("boundary of touching intervals:", boundary)


unit-disk area: pi
boundary of touching intervals: Eq(x, 0) | Eq(x, 2)


## 10. Standard regions

Specialized region objects remain convenient when the geometry is already known.


In [13]:
from semialg.standard_region_integrate import integrate_over_standard_region

interval = IntervalRegion(0, 1)
interval_integral = integrate_over_standard_region(x**2, interval, [x])
assert interval_integral == sp.Rational(1, 3)
print(interval_integral)


1/3


## 11. Higher-level exact geometry

Projection, metric queries, topology, and local algebraic geometry compose the same exact CAD/QE/optimization infrastructure.


In [14]:
projection = semialg.semialgebraic_projection(
    (x >= 0) & (x <= y) & (y <= 2), [x], [x, y]
)
distance = semialg.distance_to_region((2,), (x >= 0) & (x <= 1), [x])
convex_disk = semialg.is_convex(x**2 + y**2 <= 1, [x, y])

assert equivalent(projection, (y >= 0) & (y <= 2), [y])
assert distance == 1
assert convex_disk is True
print("projection:", projection)
print("distance:", distance)


projection: (y >= 0) & (y <= 2)
distance: 1


## 12. Symbolic region conditions and active boundaries

Region-condition conversion is structural by default. Use `eliminate=True` when exact QE/CAD is desired, and `real_parameters=True` when parameter reality should appear explicitly in the returned formula. Active-boundary strata classify which recognized inequality boundaries are simultaneously active.

In [ ]:
from semialg.symbolic_regions import (
    SemialgebraicRegion, RegionElement, region_element_conditions, as_semialgebraic_region,
)

a = sp.Symbol("a")
u = sp.Symbol("u", real=True)
mapped_region = as_semialgebraic_region(
    ParametricRegion((u,), ((u, 0, 1),), (u, u**2)), (x, y)
)
membership = RegionElement((x, y), mapped_region)

symbolic_membership = region_element_conditions(membership)
eliminated_membership = region_element_conditions(membership, eliminate=True)
assert symbolic_membership == membership
assert eliminated_membership != membership

parameter_region = SemialgebraicRegion((x >= 0) & (x <= a), (x,))
parameter_membership = RegionElement((x,), parameter_region)
real_parameter_membership = region_element_conditions(
    parameter_membership, real_parameters=True
)
assert real_parameter_membership.has(sp.Contains(a, sp.S.Reals, evaluate=False))

square = (x >= 0) & (x <= 1) & (y >= 0) & (y <= 1)
active_strata = region_active_boundary_strata(square, [x, y])
assert sum(stratum.active_count == 1 for stratum in active_strata) == 4
assert sum(stratum.active_count == 2 for stratum in active_strata) == 4

## 13. Parametric covers and reusable boundary geometry

Structural charts can certify dimension before CAD. Rich boundary results retain exact boundary cells and inclusion metadata for downstream geometry.


### Specialist namespaces

The package root is reserved for broad mathematical operations. The next examples intentionally import parameter-cover and map-degree machinery from their expert namespaces.


In [ ]:
from semialg.map_degree import parametric_map_degree
from semialg.parametric_geometry import bounded_parametric_cover

triangle = semialg.SimplexRegion(((0, 0), (1, 0), (0, 1)))
cover = bounded_parametric_cover(triangle)
assert cover.certified_dimension() == 2
assert semialg.region_dimension(triangle) == 2

boundary_info = semialg.region_boundary_result((x >= 0) & (x < 1), (x,))
assert len(boundary_info.included_strata) == 1
assert len(boundary_info.excluded_strata) == 1

affine_info = semialg.analyze_affine_map((2*x + 1, 2*y - 1), (x, y))
assert affine_info.scaled_isometry and affine_info.scale_factor == 2

map_degree = parametric_map_degree((x**2,), (x,))
assert map_degree.degree == 2

circle_distance = semialg.distance_to_region(
    (2, 0), sp.Eq(x**2 + y**2, 1), (x, y), return_result=True
)
assert circle_distance.distance == 1
assert circle_distance.optimization.method == "distance_critical_points"


## 14. Certified real algebraic feasibility and polynomial sign decisions

Specialized exact backends can avoid a full CAD when their proof conditions apply. Structured results distinguish a certified decision from an unsupported/incomplete case; fast witness searches are one-sided and every returned point is verified exactly.


In [ ]:
from semialg import (
    find_negative_point, find_negative_witness_fast,
    polynomial_nonnegative, real_algebraic_feasibility,
)
from semialg.algebraic_decomposition import (
    equidimensional_decomposition, verify_decomposition_certificate,
)

circle = real_algebraic_feasibility((x**2 + y**2 - 1,), (x, y), return_result=True)
assert circle.complete and circle.satisfiable and circle.assignment is not None
assert sp.simplify((x**2 + y**2 - 1).subs(circle.assignment)) == 0

empty = real_algebraic_feasibility((x**2 + y**2 + 1,), (x, y), return_result=True)
assert empty.complete and empty.satisfiable is False

dec = equidimensional_decomposition((x*y,), (x, y))
assert dec.complete and dec.certificate is not None
assert verify_decomposition_certificate(dec.certificate)

assert polynomial_nonnegative(x**4 + y**4 + 1, (x, y))
negative = find_negative_point(x**2 + y**2 - 1, (x, y))
fast = find_negative_witness_fast(x**3 + y**2, (x, y))
circle.assignment, dec.certificate.methods, negative, fast.assignment


## 15. What to explore next

The executable scripts under `examples/` give smaller focused demonstrations. The documentation expands on:

- CAD and variable-order strategy;
- exactness, certification, and structured `return_result=True` objects;
- function domains/ranges and certified algebraization;
- symbolic regions, topology, optimization, and integration;
- expert submodules for specialized CAD/QE and algebraic machinery.

The package intentionally distinguishes a failed/unsupported method from a mathematical negative answer: exact operations should return `unknown`/raise where appropriate rather than fabricate `True`, `False`, or an empty set.
